In [2]:
import pandas as pd
Groceries_dataset = pd.read_csv('data/Groceries_dataset.csv', sep=',', na_values=[''], quotechar='"')
Groceries_dataset

,Member_number,Date,itemDescription
0,1808,21-07-2015,tropical fruit
1,2552,05-01-2015,whole milk
2,2300,19-09-2015,pip fruit
3,1187,12-12-2015,other vegetables
4,3037,01-02-2015,whole milk
...,...,...,...
38760,4471,08-10-2014,sliced cheese
38761,2022,23-02-2014,candy
38762,1097,16-04-2014,cake bar
38763,1510,03-12-2014,fruit/vegetable juice


In [3]:

# Basic inspection
print("Shape:", Groceries_dataset.shape)
print("\nColumn names:", Groceries_dataset.columns.tolist())
print("\nData types:")
print(Groceries_dataset.dtypes)

Shape: (38765, 3)

Column names: ['Member_number', 'Date', 'itemDescription']

Data types:
Member_number       int64
Date               object
itemDescription    object
dtype: object


In [4]:
# Basic cleaning/fixes so the other cells run without errors.
# (This cell assumes pandas is available from the cells below.)

# Normalize column names
Groceries_dataset.columns = [c.strip() for c in Groceries_dataset.columns]

# Ensure required columns exist
required = {"Member_number", "Date", "itemDescription"}
missing = required - set(Groceries_dataset.columns)
if missing:
    raise KeyError(f"Missing required columns in df: {missing}")

# Parse Date column to datetime
Groceries_dataset['Date'] = pd.to_datetime(Groceries_dataset['Date'], errors='coerce')
bad_dates = Groceries_dataset['Date'].isna().sum()
if bad_dates:
    print(f"Warning: {bad_dates} rows have invalid/missing Date and were set to NaT")

# Drop rows without itemDescription (can't use them for association mining)
missing_items = Groceries_dataset['itemDescription'].isna().sum()
if missing_items:
    print(f"Dropping {missing_items} rows with missing itemDescription")
    Groceries_dataset = Groceries_dataset.dropna(subset=['itemDescription']).reset_index(drop=True)

# Normalize Member_number and fill missing
Groceries_dataset['Member_number'] = Groceries_dataset['Member_number'].fillna('Unknown').astype(str)

# Final check
print("Cleaned DataFrame shape:", Groceries_dataset.shape)
print("Date range:", Groceries_dataset['Date'].min(), "to", Groceries_dataset['Date'].max())

Cleaned DataFrame shape: (38765, 3)
Date range: 2014-01-01 00:00:00 to 2015-12-30 00:00:00


C:\Users\USER\AppData\Local\Temp\ipykernel_41564\1512687022.py:14: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  Groceries_dataset['Date'] = pd.to_datetime(Groceries_dataset['Date'], errors='coerce')


In [5]:
# Show first 10 rows
display(Groceries_dataset.head(10))

,Member_number,Date,itemDescription
0,1808,2015-07-21,tropical fruit
1,2552,2015-01-05,whole milk
2,2300,2015-09-19,pip fruit
3,1187,2015-12-12,other vegetables
4,3037,2015-02-01,whole milk
5,4941,2015-02-14,rolls/buns
6,4501,2015-05-08,other vegetables
7,3803,2015-12-23,pot plants
8,2762,2015-03-20,whole milk
9,4119,2015-02-12,tropical fruit


In [10]:
df = Groceries_dataset.copy()


# ONE-HOT ENCODING SECTION
# Create transaction ID
df['Transactions'] = df['Member_number'].astype(str) + "_" + df['Date'].astype(str)

# Group into transactions
transactions = df.groupby('Transactions')['itemDescription'].apply(list).tolist()

# One-hot encoding
from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)
encoded_data = pd.DataFrame(te_array, columns=te.columns_)

print("Encoded data shape:", encoded_data.shape)
encoded_data.head()

Encoded data shape: (14963, 167)


,Instant food products,UHT-milk,abrasive cleaner,artif. sweetener,baby cosmetics,bags,baking powder,bathroom cleaner,beef,berries,...,turkey,vinegar,waffles,whipped/sour cream,whisky,white bread,white wine,whole milk,yogurt,zwieback
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,True,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [12]:

# ASSOCIATION RULE MINING

from mlxtend.frequent_patterns import apriori, association_rules

# Frequent itemsets
frequent_itemsets = apriori(encoded_data, min_support=0.008, use_colnames=True)

# Rules
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=0.01)

# Sort results
rules = rules.sort_values(by="lift", ascending=False)

rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10)

,antecedents,consequents,support,confidence,lift
13,(sausage),(whole milk),0.008955,0.148394,0.939663
12,(whole milk),(sausage),0.008955,0.056708,0.939663
18,(whole milk),(yogurt),0.011161,0.070673,0.822940
19,(yogurt),(whole milk),0.011161,0.129961,0.822940
2,(other vegetables),(soda),0.009691,0.079365,0.817302
3,(soda),(other vegetables),0.009691,0.099794,0.817302
10,(whole milk),(rolls/buns),0.013968,0.088447,0.804028
11,(rolls/buns),(whole milk),0.013968,0.126974,0.804028
0,(other vegetables),(rolls/buns),0.010559,0.086481,0.786154
1,(rolls/buns),(other vegetables),0.010559,0.095990,0.786154
